In [1]:
import clr  # From pythonnet
import os
import random
import numpy as np
import pandas as pd
from sklearn.cluster import MeanShift
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Paths
dwsim_path = r"C:\Users\user\AppData\Local\DWSIM\\"
sim_path = r"Z:\GoogleDrive\uem\Doutorado\Ensaios\SAF\SAF_8_08_08_2025\SAF_kinetics_paper_code_with_WGS\dwsim_bench_model.dwxmz"

In [3]:
# 1. Setup paths to DWSIM installation
clr.AddReference(os.path.join(dwsim_path, "DWSIM.Automation.dll"))
clr.AddReference(os.path.join(dwsim_path, "DWSIM.Interfaces.dll"))
clr.AddReference(os.path.join(dwsim_path, "ThermoCS\\ThermoCS.dll"))

from DWSIM.Automation import Automation3
from System import String

# 2. Initialize the Automation Manager
interf = Automation3()

# 3. Load an existing simulation (.dwxmz)
Flowsheet = interf.LoadFlowsheet(sim_path)

In [4]:
syngas = Flowsheet.GetFlowsheetSimulationObject('syngas').GetAsObject()
compressor = Flowsheet.GetFlowsheetSimulationObject('C-1').GetAsObject()
cooler = Flowsheet.GetFlowsheetSimulationObject('CL-1').GetAsObject()
syncrude = Flowsheet.GetFlowsheetSimulationObject('syncrude').GetAsObject()
syncrude_phase = syncrude.GetPhase('Overall')
PFR_1 = Flowsheet.GetFlowsheetSimulationObject('PFR-1').GetAsObject()
E_reactor = Flowsheet.GetFlowsheetSimulationObject('E1').GetAsObject()
PFR_1.set_dV(0.02)

In [6]:
PFR_1.OutletTemperature

523.15

In [4]:
for item in Flowsheet.Scripts.GetEnumerator():
    item.Value.ScriptText = item.Value.ScriptText.replace("r_HCs_numerator = A_HCs*exp(-E_HCs/(R*T))*(P_H2**a)*(P_CO**b)", 
                                                          "r_HCs_numerator = A_HCs*exp(-E_HCs/(R*T))*(P_H2**a_HCs)*(P_CO**b_HCs)") 

In [5]:
for item in Flowsheet.Scripts.GetEnumerator():
    item.Value.ScriptText = item.Value.ScriptText.replace("r_HCs_denominator = (1 + k_CO*exp(-H_CO/(R*T))*(P_CO**c))**2", 
                                                          "r_HCs_denominator = (1 + k_CO*exp(-H_CO/(R*T))*(P_CO**c_HCs))**2") 

In [8]:
for item in Flowsheet.Scripts.GetEnumerator():
    item.Value.ScriptText = item.Value.ScriptText.replace("a = 0.75", "a_HCs = 0.75") 

In [11]:
for item in Flowsheet.Scripts.GetEnumerator():
    item.Value.ScriptText = item.Value.ScriptText.replace("b = 1.00", "b_HCs = 1.00") 

In [12]:
for item in Flowsheet.Scripts.GetEnumerator():
    item.Value.ScriptText = item.Value.ScriptText.replace("c = 1.00", "c_HCs = 1.00") 

interf.SaveFlowsheet(Flowsheet, sim_path, True)

In [5]:
for item in Flowsheet.Scripts.GetEnumerator():
    item.Value.ScriptText = item.Value.ScriptText.replace("thiele_220C = 4", "thiele_220C = 4") 

In [6]:
Flowsheet.Scripts['ae107d3a-c4c9-4f5f-8f73-0be01e7a36bd'].ScriptText

'from math import exp\n\nR = 8.314 # J/mol K\nn = 1\nA_HCs = 0.9550608601894232 # kmol/kg s Pa**(a+b)\nE_HCs = 142601.814307 # J/mol\nk_CO = 1.397972209918762e-05 # Pa**-c\nH_CO = 932.9790718460237 # J/mol\na = 0.75\nb = 1.00\nc = 1.00\n\nP_H2 = R1\nP_CO = R2\n\nalfa = 1.62628872e+00 - 1.49497171e-03 * T\n\ny_n = (1-alfa)*alfa**(n-1)\n\nbeta = [-13.53235038,   0.33045452,   0.02174398,   2.30345345]\nexponent = beta[0] + beta[1] * n + beta[2] * T + beta[3] * ( n == 2 )\nupsilon_n =  1 / ( 1 + ( 1 - ( ( n == 1 ) | ( n == 3 ) ) ) * exp( - exponent ) )\n\n\nr_HCs_numerator = A_HCs*exp(-E_HCs/(R*T))*(P_H2**a)*(P_CO**b)\nr_HCs_denominator = (1 + k_CO*exp(-H_CO/(R*T))*(P_CO**c))**2\n\nr_HCs = r_HCs_numerator/r_HCs_denominator\n\nfrom math import tanh\n\nthiele_220C = 4\nthiele = thiele_220C * ( ( 493 / T ) * exp( - ( E_HCs / R ) * ( 1/T - 1/493 ) ) ) ** (1/2)\neff = tanh( thiele ) / thiele\n\nr = eff*n*y_n*upsilon_n*r_HCs # kmol/kg s'

In [7]:
# v0 = 730 # L / min
# v0 = v0 / ( 1000 * 60 ) # m3 / s

# F0 = v0 * 1e5 / ( 8.314 * 273 ) # mol / s

# H2_CO_in = 2
# F_H2_in = F0 * (H2_CO_in / ( 1 + H2_CO_in )) # mol / s
# F_CO_in = F0 - F_H2_in

# H2.SetMolarFlow(F_H2_in)
# CO.SetMolarFlow(F_CO_in)
# compressor.POut = 2e6
# cooler.OutletTemperature = 300 + 273
# PFR_1.CatalystLoading = 1648

# Ligar o solver
interf.CalculateFlowsheet2(Flowsheet)

# check if solved
if not Flowsheet.Solved:
    interf.SaveFlowsheet(Flowsheet, sim_path, True)
    raise ValueError('Something went wrong at index, check your simulation.')

interf.SaveFlowsheet(Flowsheet, sim_path, True)

In [8]:
E_reactor.EnergyFlow

-12.954298486927776

In [9]:
for component in syncrude_phase.Compounds.keys():
    print(component, syncrude_phase.Compounds[component].MolarFlow)

Methane 0.004401663118093007
Ethane 0.00018146128372163479
Propane 0.0010927421125584095
N-butane 0.0002275162698705642
N-pentane 0.0002495703392967365
N-hexane 0.00026778704486480893
N-heptane 0.00028045365961116127
N-octane 0.00028635964676634
N-nonane 0.0002850852375087167
N-decane 0.0002770779938617867
N-undecane 0.0002634920543288768
N-dodecane 0.0002458750420552877
N-tridecane 0.00022583569461268174
N-tetradecane 0.0002047956179049319
N-pentadecane 0.00018386233313749358
N-hexadecane 0.0001638061909665391
N-heptadecane 0.00014510089138640867
N-octadecane 0.00012798986639109253
N-nonadecane 0.00011255380647409129
N-heneicosane 8.654238883586791e-05
N-docosane 7.575688163472284e-05
N-tetracosane 5.796146101431558e-05
N-pentacosane 5.06834035004282e-05
N-hexacosane 4.4319484118229514e-05
N-heptacosane 3.875887708330368e-05
N-octacosane 3.3902200321350486e-05
N-nonacosane 2.9661176761047567e-05
N-eicosane 9.876745512739668e-05
Water 0.08328563810193775
Hydrogen 0.18393259120577138
Ca

In [10]:
dir(PFR_1)

['AccumulationStream',
 'AccumulationStreams',
 'AddDynamicProperty',
 'AddExtraProperty',
 'AdjustVarType',
 'Annotation',
 'AppendDebugLine',
 'AttachedAdjustId',
 'AttachedSpecId',
 'AttachedUtilities',
 'Calculate',
 'Calculate_Internal',
 'Calculated',
 'CalculationRoutineOverride',
 'CanUsePreviousResults',
 'CatalystLoading',
 'CatalystParticleDiameter',
 'CatalystVoidFraction',
 'CheckDirtyStatus',
 'CheckSpec',
 'ClassId',
 'ClearExtraProperties',
 'ClearPropertyPackageInstance',
 'Clone',
 'CloneJSON',
 'CloneXML',
 'CloseDynamicsEditForm',
 'CloseEditForm',
 'ComponentConversions',
 'ComponentDescription',
 'ComponentName',
 'ConnectEnergyStream',
 'ConnectFeedEnergyStream',
 'ConnectFeedMaterialStream',
 'ConnectProductEnergyStream',
 'ConnectProductMaterialStream',
 'Conversions',
 'CopyDataToClipboard',
 'CreateChartAction',
 'CreateDimensionsList',
 'CreateDynamicProperties',
 'CreateNew',
 'DHRT',
 'DHRi',
 'DeCalculate',
 'DebugMode',
 'DebugText',
 'DeltaP',
 'DeltaQ'

In [11]:
PFR_1.GetEnergyConsumption()

-12.954298486927776

In [12]:
PFR_1.get_DeltaT()

40.0